## 🛠️ Tools in LangChain

## 1. Creating Tools (@tool)
- The simplest way to create a tool is with the @tool decorator.

In [1]:
from langchain.tools import tool
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    model="qwen/qwen3.8-27b"
)



In [6]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage


@tool
def get_weather(location: str) -> str:
    """Return the current weather for a given location."""
    return f"{location} is sunny ☀️."


llm_with_tools = llm.bind_tools([get_weather])

messages = [
    ("user", "What is the weather in Akola?")
]

# Step 1: LLM decides to call the tool
response = llm_with_tools.invoke(messages)

# Step 2: Execute tool
messages.append(response)

for tool_call in response.tool_calls:
    result = get_weather.invoke(tool_call["args"])

    messages.append(
        ToolMessage(
            content=result,
            tool_call_id=tool_call["id"]
        )
    )

# Step 3: LLM generates final answer
final_response = llm_with_tools.invoke(messages)

print(final_response.content)

The weather in Akola is currently sunny ☀️.


## 2. Custom tool name and Custom tool description 

- #### Custom Tool Name
    Override the default function-based tool name with a more descriptive name.

- #### Custom Tool Description
    Override the automatically generated description to provide clearer guidance to the model.

In [ ]:
from langchain_core.tools import tool
from langchain.messages import HumanMessage, ToolMessage


@tool(
    "web_search",
    description="Use this tool to get the current weather for a given location."
)
def get_weather2(location: str) -> str:
    """Return the weather for a location."""
    return f"The weather for {location} is rainy."


llm_with_tools = llm.bind_tools([get_weather2])

messages = [
    HumanMessage(content="What is the weather in Pune?")
]

# 1. LLM decides which tool to call
response = llm_with_tools.invoke(messages)

# 2. Add the AI response to conversation history
messages.append(response)

# 3. Execute the requested tool
for tool_call in response.tool_calls:
    tool_result = get_weather2.invoke(tool_call["args"])

    messages.append(
        ToolMessage(
            content=tool_result,
            tool_call_id=tool_call["id"]
        )
    )

# 4. Send tool result back to LLM
final = llm_with_tools.invoke(messages)

print(final.content)

Tool Name: web_search
Arguments: {'location': 'Pune'}
Tool Description: Use this tool to get the current weather for a given location.
Tool Result: The weather for Pune is rainy.
Final Answer: It's currently rainy in Pune. If you're heading out, it might be a good idea to have an umbrella or raincoat with you!


### Input Schema & Validation (Pydantic / JSON Schema)

In [10]:

from pydantic import BaseModel, Field
from typing import Literal

from langchain.tools import tool
from langchain.messages import ToolMessage

# 1. Define the input schema
class CalculationInput(BaseModel):
    num1: int = Field(description="First number")
    num2: int = Field(description="Second number")
    operator: Literal["+", "-", "/", "*"] = Field(
        description="Mathematical operator"
    )


# 2. Create the calculator tool
@tool(
    "calculator",
    description="Perform arithmetic calculations using two numbers.",
    args_schema=CalculationInput,
)
def calculator(
    num1: int,
    num2: int,
    operator: Literal["+", "-", "/", "*"] = "+"
) -> str:

    if operator == "+":
        result = num1 + num2
    elif operator == "-":
        result = num1 - num2
    elif operator == "*":
        result = num1 * num2
    elif operator == "/":
        if num2 == 0:
            return "Error: Division by zero is not allowed."
        result = num1 / num2

    return f"The calculation of {num1} {operator} {num2} is: {result}"


# 3. Bind the tool to your LLM
llm_with_tools = llm.bind_tools([calculator])

# 4. Send a user message
messages = [
    ("user", "What is the addition of 5 + 6?")
]

response = llm_with_tools.invoke(messages)

# 5. Execute requested tools
messages.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == "calculator":
        result = calculator.invoke(tool_call["args"])

        messages.append(
            ToolMessage(
                content=result,
                tool_call_id=tool_call["id"],
            )
        )

# 6. Ask the LLM to produce the final response
final_result = llm_with_tools.invoke(messages)

print(final_result.content)


The addition of 5 + 6 is **11**.
